In [1]:
%load_ext autoreload
%autoreload 2

import os, sys

sys.path.insert(0, "../..")
from progsnap2.spec.enums import MainTableColumns as Cols, EventType

import pandas as pd

In [2]:
base_dir = '../../../sample_data/edwards/raw/2019/'

In [3]:
keystrokes = pd.read_csv(os.path.join(base_dir, 'keystrokes.csv'))
keystrokes.head()

C:\Users\twprice\AppData\Local\Temp\ipykernel_33644\3859181010.py:1: DtypeWarning: Columns (12,13,14,15) have mixed types. Specify dtype option on import or set low_memory=False.
  keystrokes = pd.read_csv(os.path.join(base_dir, 'keystrokes.csv'))


,EventID,SubjectID,AssignmentID,CodeStateSection,X-Task,EventType,X-Keystroke,InsertText,DeleteText,SourceLocation,ClientTimestamp,EditType,X-RunInput,X-RunOutput,X-RunHasError,X-RunUserTerminated,X-RawAssignmentID,X-Term,X-Compilable
0,0,S000,p4f,task0.py,0,File.Edit,#,#,NaN,0.0,1568242704669,Insert,NaN,NaN,NaN,NaN,p4,f,1
1,1,S000,p4f,task0.py,0,File.Edit,space,,NaN,1.0,1568242704830,Insert,NaN,NaN,NaN,NaN,p4,f,1
2,2,S000,p4f,task0.py,0,File.Edit,delete,NaN,,1.0,1568242705198,Delete,NaN,NaN,NaN,NaN,p4,f,1
3,3,S000,p4f,task0.py,0,File.Edit,space,,NaN,1.0,1568242705450,Insert,NaN,NaN,NaN,NaN,p4,f,1
4,4,S000,p4f,task0.py,0,File.Edit,@,@,NaN,2.0,1568242705692,Insert,NaN,NaN,NaN,NaN,p4,f,1


In [6]:
keystrokes.EventType.value_counts()

EventType
File.Edit       5039755
Run.Program       75112
X-SwitchTask      12104
Submit             3326
Name: count, dtype: int64

In [7]:
keystrokes[keystrokes["X-Compilable"] == True]["X-RunHasError"].value_counts()

X-RunHasError
False    19547
True      4011
Name: count, dtype: int64

In [8]:
keystrokes[["X-RunHasError", "X-Compilable"]].value_counts()

X-RunHasError  X-Compilable
False          1               19547
True           0                7564
               1                4011
False          0                1875
Name: count, dtype: int64

In [9]:
def get_error_type(row):
    if row["X-Compilable"] == 0:
          return "SyntaxError"
    if row["X-RunHasError"] == True:
        return "Runtime Error"
    else:
        raise ValueError("Row does not have an error")

In [10]:
errors = keystrokes[(keystrokes[Cols.EventType] == EventType.RunProgram) & (keystrokes["X-RunHasError"] | (keystrokes["X-Compilable"] == 0))].copy()
errors[Cols.CompileMessageType] = errors.apply(get_error_type, axis=1)
errors

,EventID,SubjectID,AssignmentID,CodeStateSection,X-Task,EventType,X-Keystroke,InsertText,DeleteText,SourceLocation,ClientTimestamp,EditType,X-RunInput,X-RunOutput,X-RunHasError,X-RunUserTerminated,X-RawAssignmentID,X-Term,X-Compilable,CompileMessageType
8263,17015,S002,p4f,task1.py,1,Run.Program,NaN,NaN,NaN,NaN,1568358538569,NaN,[],"[""SyntaxError: bad input on line 7""]",True,False,p4,f,0,SyntaxError
8445,17197,S002,p4f,task1.py,1,Run.Program,NaN,NaN,NaN,NaN,1568358779040,NaN,[],"[""SyntaxError: bad input on line 11""]",True,False,p4,f,0,SyntaxError
8447,17199,S002,p4f,task1.py,1,Run.Program,NaN,NaN,NaN,NaN,1568358788939,NaN,[],"[""SyntaxError: bad input on line 11""]",True,False,p4,f,0,SyntaxError
8451,17203,S002,p4f,task1.py,1,Run.Program,NaN,NaN,NaN,NaN,1568358804780,NaN,[],"[""SyntaxError: bad input on line 11""]",True,False,p4,f,0,SyntaxError
8459,17211,S002,p4f,task1.py,1,Run.Program,NaN,NaN,NaN,NaN,1568358817854,NaN,[],"[""SyntaxError: bad input on line 11""]",True,False,p4,f,0,SyntaxError
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5129297,9181762,S516,p8f,task1.py,1,Run.Program,NaN,NaN,NaN,NaN,1570829355186,NaN,[],"[""SyntaxError: bad input on line 48""]",True,False,p8,f,0,SyntaxError
5129298,9181763,S516,p8f,task1.py,1,Run.Program,NaN,NaN,NaN,NaN,1570829359590,NaN,[],"[""SyntaxError: bad input on line 48""]",True,False,p8,f,0,SyntaxError
5129303,9181769,S516,p8f,task1.py,1,Run.Program,NaN,NaN,NaN,NaN,1570833508356,NaN,[],"[""SyntaxError: bad input on line 41""]",True,False,p8,f,0,SyntaxError
5129522,9181989,S516,p8f,task1.py,1,Run.Program,NaN,NaN,NaN,NaN,1570837312800,NaN,"[""stay""]","[""Your Door is #1!\n"", ""But behind Door #2 is ...",True,False,p8,f,1,Runtime Error


In [11]:
errors.CompileMessageType.value_counts()

CompileMessageType
SyntaxError      9439
Runtime Error    4011
Name: count, dtype: int64

In [12]:
errors[Cols.EventType] = EventType.CompileError
errors[Cols.ParentEventID] = errors[Cols.EventID].astype(str)
errors[Cols.EventID] = errors[Cols.EventID].astype(str) + "_parent"
errors

,EventID,SubjectID,AssignmentID,CodeStateSection,X-Task,EventType,X-Keystroke,InsertText,DeleteText,SourceLocation,...,EditType,X-RunInput,X-RunOutput,X-RunHasError,X-RunUserTerminated,X-RawAssignmentID,X-Term,X-Compilable,CompileMessageType,ParentEventID
8263,17015_parent,S002,p4f,task1.py,1,Compile.Error,NaN,NaN,NaN,NaN,...,NaN,[],"[""SyntaxError: bad input on line 7""]",True,False,p4,f,0,SyntaxError,17015
8445,17197_parent,S002,p4f,task1.py,1,Compile.Error,NaN,NaN,NaN,NaN,...,NaN,[],"[""SyntaxError: bad input on line 11""]",True,False,p4,f,0,SyntaxError,17197
8447,17199_parent,S002,p4f,task1.py,1,Compile.Error,NaN,NaN,NaN,NaN,...,NaN,[],"[""SyntaxError: bad input on line 11""]",True,False,p4,f,0,SyntaxError,17199
8451,17203_parent,S002,p4f,task1.py,1,Compile.Error,NaN,NaN,NaN,NaN,...,NaN,[],"[""SyntaxError: bad input on line 11""]",True,False,p4,f,0,SyntaxError,17203
8459,17211_parent,S002,p4f,task1.py,1,Compile.Error,NaN,NaN,NaN,NaN,...,NaN,[],"[""SyntaxError: bad input on line 11""]",True,False,p4,f,0,SyntaxError,17211
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5129297,9181762_parent,S516,p8f,task1.py,1,Compile.Error,NaN,NaN,NaN,NaN,...,NaN,[],"[""SyntaxError: bad input on line 48""]",True,False,p8,f,0,SyntaxError,9181762
5129298,9181763_parent,S516,p8f,task1.py,1,Compile.Error,NaN,NaN,NaN,NaN,...,NaN,[],"[""SyntaxError: bad input on line 48""]",True,False,p8,f,0,SyntaxError,9181763
5129303,9181769_parent,S516,p8f,task1.py,1,Compile.Error,NaN,NaN,NaN,NaN,...,NaN,[],"[""SyntaxError: bad input on line 41""]",True,False,p8,f,0,SyntaxError,9181769
5129522,9181989_parent,S516,p8f,task1.py,1,Compile.Error,NaN,NaN,NaN,NaN,...,NaN,"[""stay""]","[""Your Door is #1!\n"", ""But behind Door #2 is ...",True,False,p8,f,1,Runtime Error,9181989


In [13]:
# Errors seem to just be syntax errors, no need to parse the output

# def parse_error(ouput):
#     try:
#         lst = eval(ouput)
#         return lst[-1]
#     except Exception as e:
#         return None

# errors[errors["X-Compilable"] == False]["X-RunOutput"].apply(parse_error).str[:5].value_counts()

In [14]:
keystrokes[Cols.CompileMessageType] = pd.NA
keystrokes[Cols.ParentEventID] = pd.NA
keystrokes[Cols.EventID] = keystrokes[Cols.EventID].astype(str)

In [15]:
events_with_errors = pd.concat([keystrokes, errors], ignore_index=True)

In [16]:
events_with_errors[events_with_errors.CompileMessageType.notna()].head()

,EventID,SubjectID,AssignmentID,CodeStateSection,X-Task,EventType,X-Keystroke,InsertText,DeleteText,SourceLocation,...,EditType,X-RunInput,X-RunOutput,X-RunHasError,X-RunUserTerminated,X-RawAssignmentID,X-Term,X-Compilable,CompileMessageType,ParentEventID
5130297,17015_parent,S002,p4f,task1.py,1,Compile.Error,NaN,NaN,NaN,NaN,...,NaN,[],"[""SyntaxError: bad input on line 7""]",True,False,p4,f,0,SyntaxError,17015
5130298,17197_parent,S002,p4f,task1.py,1,Compile.Error,NaN,NaN,NaN,NaN,...,NaN,[],"[""SyntaxError: bad input on line 11""]",True,False,p4,f,0,SyntaxError,17197
5130299,17199_parent,S002,p4f,task1.py,1,Compile.Error,NaN,NaN,NaN,NaN,...,NaN,[],"[""SyntaxError: bad input on line 11""]",True,False,p4,f,0,SyntaxError,17199
5130300,17203_parent,S002,p4f,task1.py,1,Compile.Error,NaN,NaN,NaN,NaN,...,NaN,[],"[""SyntaxError: bad input on line 11""]",True,False,p4,f,0,SyntaxError,17203
5130301,17211_parent,S002,p4f,task1.py,1,Compile.Error,NaN,NaN,NaN,NaN,...,NaN,[],"[""SyntaxError: bad input on line 11""]",True,False,p4,f,0,SyntaxError,17211


In [17]:
errors.AssignmentID.value_counts()

AssignmentID
p8f    6128
p7f    3013
p6f    2815
p4f    1106
p5f     388
Name: count, dtype: int64

In [18]:
grades = pd.read_csv(os.path.join(base_dir, 'students.csv'))
link_subject_path = os.path.join(base_dir, '..', '..', '2019', 'LinkTables', 'SubjectTerm.csv')
# create dir if not exists
os.makedirs(os.path.dirname(link_subject_path), exist_ok=True)

grades.head()

,SubjectID,Group,SyntaxExercises,p4,p5,p6,p7,p8,exam1,exam2
0,S275,Spring,False,100.0,100,95.0,96.0,100.0,78.0,60.0
1,S047,Spring,False,100.0,100,95.0,100.0,100.0,82.0,80.0
2,S503,Spring,False,100.0,100,100.0,95.0,45.0,84.0,68.0
3,S492,Spring,False,100.0,100,100.0,100.0,100.0,94.0,86.0
4,S300,Spring,False,50.0,40,80.0,90.0,53.0,54.0,50.0


In [19]:
score_rows = grades.drop(columns=[Cols.SubjectID, "Group", "SyntaxExercises"])
shifted = score_rows.shift(axis=0)
same_as_last = (score_rows == shifted)
same_as_last = same_as_last.all(axis=1)

In [20]:
duplicated_subject_ids = grades[grades[[Cols.SubjectID, "Group"]].duplicated()][Cols.SubjectID].unique()
duplicated_subject_ids


array([], dtype=object)

In [21]:
grades_deduped = grades[~same_as_last & ~grades[Cols.SubjectID].isin(duplicated_subject_ids)].copy()
grades_deduped.rename(columns={"Group": Cols.TermID}, inplace=True)

grades_deduped.to_csv(link_subject_path, index=False)

In [22]:
grades.shape[0] - grades_deduped.shape[0]

0

In [34]:
grades_deduped[[Cols.SubjectID, Cols.TermID]].duplicated().sum()

np.int64(0)

In [35]:
assignment_cols = [f"p{i}" for i in range(4, 8)]
assignment_grades = None
for semester in ["Fall", "Spring"]:
    semester_grades = grades_deduped[grades_deduped[Cols.TermID] == semester]
    melted_grades = semester_grades[[Cols.SubjectID] + assignment_cols].melt(id_vars=Cols.SubjectID, var_name=Cols.AssignmentID, value_name=Cols.Score)
    melted_grades[Cols.TermID] = semester
    melted_grades[Cols.AssignmentID] += semester[0].lower()  # e.g. p4 -> p4f, p5 -> p5f
    if assignment_grades is None:
        assignment_grades = melted_grades
    else:
        assignment_grades = pd.concat([assignment_grades, melted_grades], ignore_index=True)

assignment_grades

,SubjectID,AssignmentID,Score,TermID
0,S438,p4f,95.0,Fall
1,S119,p4f,100.0,Fall
2,S198,p4f,90.0,Fall
3,S436,p4f,95.0,Fall
4,S474,p4f,100.0,Fall
...,...,...,...,...
1287,S298,p7s,100.0,Spring
1288,S097,p7s,0.0,Spring
1289,S380,p7s,95.0,Spring
1290,S225,p7s,100.0,Spring


In [36]:
assert assignment_grades[[Cols.AssignmentID, Cols.SubjectID]].value_counts().max() == 1

In [37]:
import numpy as np

submissions = keystrokes[keystrokes[Cols.EventType] == EventType.Submit]
last_submissions = submissions.loc[submissions.groupby([Cols.SubjectID, Cols.AssignmentID])[Cols.ClientTimestamp].idxmax()]
last_submissions = last_submissions[[Cols.EventID, Cols.AssignmentID, Cols.SubjectID]].copy()
last_submissions


,EventID,AssignmentID,SubjectID
1944,1948,p4s,S001
3075,3096,p5s,S001
7970,11799,p7s,S001
8622,17374,p4f,S002
10374,19126,p5f,S002
...,...,...,...
5119604,9172061,p6f,S516
5119034,9171487,p6s,S516
5123215,9175676,p7f,S516
5123210,9175671,p7s,S516


In [38]:
last_submissions.groupby([Cols.SubjectID, Cols.AssignmentID]).EventID.count().max()

np.int64(1)

In [39]:
last_submissions.AssignmentID.value_counts()

AssignmentID
p5s    180
p7f    179
p7s    176
p5f    170
p4f    170
p4s    147
p6f    137
p8s    128
p6s    123
p8f    109
Name: count, dtype: int64

In [40]:
last_submissions_with_grades = last_submissions.merge(assignment_grades, on=[Cols.SubjectID, Cols.AssignmentID], how='inner')
last_submissions_with_grades

,EventID,AssignmentID,SubjectID,Score,TermID
0,1948,p4s,S001,100.0,Spring
1,3096,p5s,S001,100.0,Spring
2,11799,p7s,S001,100.0,Spring
3,17374,p4f,S002,90.0,Fall
4,19126,p5f,S002,90.0,Fall
...,...,...,...,...,...
797,9100890,p7f,S512,100.0,Fall
798,9114397,p5f,S513,80.0,Fall
799,9118745,p6f,S513,100.0,Fall
800,9123733,p7f,S513,100.0,Fall


In [41]:
keys_grades = set(assignment_grades[[Cols.SubjectID, Cols.AssignmentID]].itertuples(index=False, name=None))
keys_submissions = set(last_submissions[[Cols.SubjectID, Cols.AssignmentID]].itertuples(index=False, name=None))

print(keys_grades - keys_submissions)
print(keys_submissions - keys_grades)

{('S345', 'p7f'), ('S296', 'p7s'), ('S077', 'p6f'), ('S122', 'p6f'), ('S327', 'p7s'), ('S205', 'p4s'), ('S455', 'p7f'), ('S482', 'p4f'), ('S373', 'p4s'), ('S048', 'p6s'), ('S133', 'p6f'), ('S440', 'p5s'), ('S007', 'p6s'), ('S056', 'p4f'), ('S224', 'p5s'), ('S400', 'p6f'), ('S039', 'p7s'), ('S133', 'p5f'), ('S482', 'p6f'), ('S292', 'p6f'), ('S047', 'p7s'), ('S199', 'p6s'), ('S046', 'p6s'), ('S399', 'p6f'), ('S272', 'p4f'), ('S300', 'p4s'), ('S167', 'p6s'), ('S400', 'p5f'), ('S489', 'p5s'), ('S482', 'p5f'), ('S292', 'p5f'), ('S052', 'p7s'), ('S399', 'p5f'), ('S147', 'p7f'), ('S440', 'p4s'), ('S503', 'p6s'), ('S093', 'p6f'), ('S045', 'p6f'), ('S224', 'p4s'), ('S028', 'p4f'), ('S258', 'p5f'), ('S039', 'p5s'), ('S243', 'p6f'), ('S417', 'p5s'), ('S054', 'p6f'), ('S047', 'p5s'), ('S091', 'p7s'), ('S269', 'p7s'), ('S045', 'p5f'), ('S508', 'p5f'), ('S405', 'p6s'), ('S112', 'p7f'), ('S427', 'p6s'), ('S164', 'p4f'), ('S033', 'p4f'), ('S373', 'p6s'), ('S439', 'p6s'), ('S239', 'p5s'), ('S039', 'p4s

In [42]:
keystrokes.groupby([Cols.SubjectID, Cols.AssignmentID]).apply(lambda x: sum(x[Cols.EventType] == "Submit")).value_counts()

C:\Users\twprice\AppData\Local\Temp\ipykernel_41060\3522922229.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  keystrokes.groupby([Cols.SubjectID, Cols.AssignmentID]).apply(lambda x: sum(x[Cols.EventType] == "Submit")).value_counts()


1     717
2     424
3     156
4      83
0      63
5      51
6      29
7      19
8      14
10     10
9       7
11      3
14      3
17      1
19      1
13      1
Name: count, dtype: int64

In [43]:
events_with_errors_and_submits = events_with_errors.merge(
    last_submissions_with_grades[[Cols.EventID, "Score"]],
    on=[Cols.EventID], how='left')
events_with_errors_and_submits

,EventID,SubjectID,AssignmentID,CodeStateSection,X-Task,EventType,X-Keystroke,InsertText,DeleteText,SourceLocation,...,X-RunInput,X-RunOutput,X-RunHasError,X-RunUserTerminated,X-RawAssignmentID,X-Term,X-Compilable,CompileMessageType,ParentEventID,Score
0,0,S000,p4f,task0.py,0,File.Edit,#,#,NaN,0.0,...,NaN,NaN,NaN,NaN,p4,f,1,NaN,NaN,NaN
1,1,S000,p4f,task0.py,0,File.Edit,space,,NaN,1.0,...,NaN,NaN,NaN,NaN,p4,f,1,NaN,NaN,NaN
2,2,S000,p4f,task0.py,0,File.Edit,delete,NaN,,1.0,...,NaN,NaN,NaN,NaN,p4,f,1,NaN,NaN,NaN
3,3,S000,p4f,task0.py,0,File.Edit,space,,NaN,1.0,...,NaN,NaN,NaN,NaN,p4,f,1,NaN,NaN,NaN
4,4,S000,p4f,task0.py,0,File.Edit,@,@,NaN,2.0,...,NaN,NaN,NaN,NaN,p4,f,1,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5143742,9181762_parent,S516,p8f,task1.py,1,Compile.Error,NaN,NaN,NaN,NaN,...,[],"[""SyntaxError: bad input on line 48""]",True,False,p8,f,0,SyntaxError,9181762,NaN
5143743,9181763_parent,S516,p8f,task1.py,1,Compile.Error,NaN,NaN,NaN,NaN,...,[],"[""SyntaxError: bad input on line 48""]",True,False,p8,f,0,SyntaxError,9181763,NaN
5143744,9181769_parent,S516,p8f,task1.py,1,Compile.Error,NaN,NaN,NaN,NaN,...,[],"[""SyntaxError: bad input on line 41""]",True,False,p8,f,0,SyntaxError,9181769,NaN
5143745,9181989_parent,S516,p8f,task1.py,1,Compile.Error,NaN,NaN,NaN,NaN,...,"[""stay""]","[""Your Door is #1!\n"", ""But behind Door #2 is ...",True,False,p8,f,1,Runtime Error,9181989,NaN


In [44]:
events_with_errors_and_submits[events_with_errors_and_submits.EventType == "Submit"].Score.notna().value_counts()

Score
False    2524
True      802
Name: count, dtype: int64

In [45]:
# 403 attempts have not submission grade
# This is in small part (63) because there was no submit event
# Much larger part is due to dropping students from the gradebook
# But these will have no label anyway...
events_with_errors_and_submits.groupby([Cols.SubjectID, Cols.AssignmentID]).Score.apply(lambda x: sum(x.notna())).value_counts()

Score
1    802
0    780
Name: count, dtype: int64

In [46]:
events_with_errors_and_submits[Cols.AssignmentID].isna().sum()

np.int64(0)

In [48]:
events_with_errors_and_submits[Cols.TermID] = events_with_errors_and_submits[Cols.AssignmentID].str[-1].map(lambda x: "Spring" if x == "s" else "Fall")
events_with_errors_and_submits[Cols.TermID].value_counts()

TermID
Fall      2814469
Spring    2329278
Name: count, dtype: int64

In [49]:
events_with_errors_and_submits[Cols.CompileMessageType].value_counts()

CompileMessageType
SyntaxError      9439
Runtime Error    4011
Name: count, dtype: int64

In [ ]:
events_with_errors_and_submits.to_csv(os.path.join(base_dir, '..', '..', '2019', 'MainTable.csv'), index=False)

In [ ]:
def reconstruct(df):
    s = ''
    for _,row in df[df.EventType=='File.Edit'].iterrows():
        i = int(row.SourceLocation)
        insert = '' if pd.isna(row.InsertText) else row.InsertText
        delete = '' if pd.isna(row.DeleteText) else row.DeleteText
        s = s[:i] + insert + s[i+len(delete):]
    return s

In [6]:
def reconstruct_all(df, in_place=False):
    if not in_place:
        df = df.copy()
    codestates = []
    s = ''
    for _,row in df.iterrows():
        if row.EventType == 'File.Edit':
            i = int(row.SourceLocation)
            insert = '' if pd.isna(row.InsertText) else row.InsertText
            delete = '' if pd.isna(row.DeleteText) else row.DeleteText
            s = s[:i] + insert + s[i+len(delete):]
        codestates.append(s)
    df['CodeState'] = codestates
    return df

In [7]:
one_attempt = keystrokes[(keystrokes.SubjectID == "S000") & (keystrokes.AssignmentID == "p4f")]
one_attempt

,EventID,SubjectID,AssignmentID,CodeStateSection,X-Task,EventType,X-Keystroke,InsertText,DeleteText,SourceLocation,ClientTimestamp,EditType,X-RunInput,X-RunOutput,X-RunHasError,X-RunUserTerminated,X-RawAssignmentID,X-Term,X-Compilable
0,0,S000,p4f,task0.py,0,File.Edit,#,#,NaN,0.0,1568242704669,Insert,NaN,NaN,NaN,NaN,p4,f,1
1,1,S000,p4f,task0.py,0,File.Edit,space,,NaN,1.0,1568242704830,Insert,NaN,NaN,NaN,NaN,p4,f,1
2,2,S000,p4f,task0.py,0,File.Edit,delete,NaN,,1.0,1568242705198,Delete,NaN,NaN,NaN,NaN,p4,f,1
3,3,S000,p4f,task0.py,0,File.Edit,space,,NaN,1.0,1568242705450,Insert,NaN,NaN,NaN,NaN,p4,f,1
4,4,S000,p4f,task0.py,0,File.Edit,@,@,NaN,2.0,1568242705692,Insert,NaN,NaN,NaN,NaN,p4,f,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1896,1896,S000,p4f,task0.py,0,File.Edit,t,t,NaN,865.0,1568244374034,Insert,NaN,NaN,NaN,NaN,p4,f,0
1897,1897,S000,p4f,task0.py,0,File.Edit,space,,NaN,866.0,1568244374086,Insert,NaN,NaN,NaN,NaN,p4,f,0
1898,1898,S000,p4f,task0.py,0,File.Edit,i,i,NaN,867.0,1568244374182,Insert,NaN,NaN,NaN,NaN,p4,f,0
1899,1899,S000,p4f,task0.py,0,File.Edit,t,t,NaN,868.0,1568244374297,Insert,NaN,NaN,NaN,NaN,p4,f,0


In [8]:
with_codestate = reconstruct_all(one_attempt)

with_codestate

,EventID,SubjectID,AssignmentID,CodeStateSection,X-Task,EventType,X-Keystroke,InsertText,DeleteText,SourceLocation,ClientTimestamp,EditType,X-RunInput,X-RunOutput,X-RunHasError,X-RunUserTerminated,X-RawAssignmentID,X-Term,X-Compilable,CodeState
0,0,S000,p4f,task0.py,0,File.Edit,#,#,NaN,0.0,1568242704669,Insert,NaN,NaN,NaN,NaN,p4,f,1,#
1,1,S000,p4f,task0.py,0,File.Edit,space,,NaN,1.0,1568242704830,Insert,NaN,NaN,NaN,NaN,p4,f,1,#
2,2,S000,p4f,task0.py,0,File.Edit,delete,NaN,,1.0,1568242705198,Delete,NaN,NaN,NaN,NaN,p4,f,1,#
3,3,S000,p4f,task0.py,0,File.Edit,space,,NaN,1.0,1568242705450,Insert,NaN,NaN,NaN,NaN,p4,f,1,#
4,4,S000,p4f,task0.py,0,File.Edit,@,@,NaN,2.0,1568242705692,Insert,NaN,NaN,NaN,NaN,p4,f,1,# @
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1896,1896,S000,p4f,task0.py,0,File.Edit,t,t,NaN,865.0,1568244374034,Insert,NaN,NaN,NaN,NaN,p4,f,0,# @@@@@@@@@@@@@@@\n# But you can call me @@@@\...
1897,1897,S000,p4f,task0.py,0,File.Edit,space,,NaN,866.0,1568244374086,Insert,NaN,NaN,NaN,NaN,p4,f,0,# @@@@@@@@@@@@@@@\n# But you can call me @@@@\...
1898,1898,S000,p4f,task0.py,0,File.Edit,i,i,NaN,867.0,1568244374182,Insert,NaN,NaN,NaN,NaN,p4,f,0,# @@@@@@@@@@@@@@@\n# But you can call me @@@@\...
1899,1899,S000,p4f,task0.py,0,File.Edit,t,t,NaN,868.0,1568244374297,Insert,NaN,NaN,NaN,NaN,p4,f,0,# @@@@@@@@@@@@@@@\n# But you can call me @@@@\...


In [ ]:
for i in range(len(with_codestate)):
    if i % 100 == 0:
        print(with_codestate.iloc[i].CodeState)
        print('\n-------\n')

#

-------

# @@@@@@@@@@@@@@@
# Assn # 4
#creates a snowman that technically fil

-------

# @@@@@@@@@@@@@@@
# Assn # 4
#creates a snowman that technically fills all the points in the rubric
#his name is legalese-man
import turtle,
def create_circle()

-------

# @@@@@@@@@@@@@@@
# Assn # 4
#creates a snowman that technically fills all the points in the rubric
#his name is legalese-man
import turtle
def createCircl

-------

# @@@@@@@@@@@@@@@
# Assn # 4
#creates a snowman that technically meets all the points in the rubric
#his name is legalese-man
import turtle

#create a function crea
def createCircle

-------

# @@@@@@@@@@@@@@@
# Assn # 4
#creates a snowman that technically meets all the points in the rubric
#his name is legalese-man
import turtle

#create a function creates a circle.  Takes multiple sizes, and can be filled or not.  Also a color 
def createCircle(Size, col):

-------

# @@@@@@@@@@@@@@@
# Assn # 4
#creates a snowman that technically meets all the points in the rubric

In [ ]:
from progsnap2.spec.enums import MainTableColumns as Cols
from progsnap2.spec.spec_definition import PS2Versions
import pandas as pd